# 🧬 Chromatin Factor Paralog Synthetic Lethality Scanner
## DepMap Public 25Q3 - OLS Confounder Adjustment

---

### 概要
DepMap Public 25Q3 データを用いた、クロマチンリモデリング因子パラログの合成致死性一括スキャンパイプライン

### 特徴
- **SWI/SNF, PRC, NuRD, INO80, ISWI** 等のクロマチン複合体パラログを網羅（60ペア）
- **statsmodels OLS** による交絡補正（組織、発現量、コピー数）
- **多重検定補正**（FDR, Benjamini-Hochberg法）
- バッチ処理による効率的な全ペア解析

### 参考文献
1. Tsherniak A, et al. Cell 2017 - DepMap
2. Hoffman GR, et al. PNAS 2014 - SMARCA4/SMARCA2
3. Helming KC, et al. Cancer Cell 2014 - ARID1A/ARID1B

## 1. セットアップ

In [ ]:
# 必要なパッケージのインストール
!pip install statsmodels -q

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
from scipy.stats import false_discovery_control
import statsmodels.api as sm
import statsmodels.formula.api as smf
from typing import List, Dict, Tuple, Optional, Set
from dataclasses import dataclass, field
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# 日本語フォント設定（オプション）
# !pip install japanize-matplotlib -q
# import japanize_matplotlib

print("Setup complete!")

## 2. DepMap 25Q3 データのダウンロード

DepMap Portal からデータをダウンロードします。

**必要なファイル:**
| ファイル | 説明 | サイズ |
|---------|------|--------|
| CRISPRGeneEffect.csv | CRISPR依存性スコア | ~200MB |
| Model.csv | 細胞株メタデータ | ~5MB |
| OmicsSomaticMutations.csv | 体細胞変異 | ~500MB |
| OmicsExpressionProteinCodingGenesTPMLogp1.csv | 発現データ | ~100MB |
| OmicsCNGene.csv | コピー数データ | ~50MB |

In [ ]:
# データディレクトリの作成
!mkdir -p depmap_25q3

# DepMap 25Q3 データのダウンロード
# 注意: URLは最新版に更新してください（https://depmap.org/portal/download/all/）

BASE_URL = "https://figshare.com/ndownloader/files"  # DepMap figshare

# 25Q3のファイルID（最新版は DepMap Portal で確認）
# 以下は例です。実際のファイルIDに置き換えてください。

FILES = {
    # 'CRISPRGeneEffect.csv': 'FILE_ID_HERE',
    # 'Model.csv': 'FILE_ID_HERE',
}

print("""\n⚠️  データダウンロードの手順:

1. https://depmap.org/portal/download/all/ にアクセス
2. 'DepMap Public 25Q3' を選択
3. 以下のファイルをダウンロード:
   - CRISPRGeneEffect.csv
   - Model.csv  
   - OmicsSomaticMutations.csv
   - OmicsExpressionProteinCodingGenesTPMLogp1.csv
   - OmicsCNGene.csv (optional)
4. Colab の 'depmap_25q3' フォルダにアップロード

または、Google Drive にデータを置いてマウントすることも可能です。
""")

In [ ]:
# Google Drive からデータを読み込む場合（オプション）
USE_GOOGLE_DRIVE = False  # True に変更して使用

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = '/content/drive/MyDrive/depmap_25q3'  # パスを調整
else:
    DATA_DIR = './depmap_25q3'

## 3. クロマチン因子パラログデータベース

In [ ]:
@dataclass
class ChromatinParalogPair:
    """クロマチン因子パラログペア"""
    gene_a: str           # 変異/低発現遺伝子（ドライバー）
    gene_b: str           # 依存性ターゲット（パラログ）
    complex_name: str     # 所属複合体
    subunit_type: str     # サブユニットタイプ
    evidence_level: str   # エビデンスレベル
    pmid: str = ""        # 参考文献PMID


# クロマチンリモデリング複合体のパラログペア（文献ベース）
CHROMATIN_PARALOG_DATABASE = [
    # ========== SWI/SNF (BAF/PBAF) Complex ==========
    ChromatinParalogPair("SMARCA4", "SMARCA2", "SWI/SNF", "ATPase", "published", "26552009"),
    ChromatinParalogPair("SMARCA2", "SMARCA4", "SWI/SNF", "ATPase", "published", "26552009"),
    ChromatinParalogPair("ARID1A", "ARID1B", "SWI/SNF (BAF)", "DNA-binding", "published", "24520176"),
    ChromatinParalogPair("ARID1B", "ARID1A", "SWI/SNF (BAF)", "DNA-binding", "published", "24520176"),
    ChromatinParalogPair("ARID2", "ARID1A", "SWI/SNF (PBAF)", "DNA-binding", "predicted", ""),
    ChromatinParalogPair("BCL7A", "BCL7B", "SWI/SNF", "Accessory", "predicted", ""),
    ChromatinParalogPair("BCL7B", "BCL7C", "SWI/SNF", "Accessory", "predicted", ""),
    ChromatinParalogPair("BRD9", "BRD7", "SWI/SNF (ncBAF/PBAF)", "Bromodomain", "published", "31253568"),
    ChromatinParalogPair("BRD7", "BRD9", "SWI/SNF (PBAF/ncBAF)", "Bromodomain", "published", "31253568"),
    ChromatinParalogPair("DPF1", "DPF2", "SWI/SNF (BAF)", "PHD-finger", "predicted", ""),
    ChromatinParalogPair("DPF2", "DPF3", "SWI/SNF (BAF)", "PHD-finger", "predicted", ""),
    ChromatinParalogPair("SMARCC1", "SMARCC2", "SWI/SNF", "Core", "predicted", ""),
    ChromatinParalogPair("SMARCC2", "SMARCC1", "SWI/SNF", "Core", "predicted", ""),
    ChromatinParalogPair("SMARCD1", "SMARCD2", "SWI/SNF", "Core", "predicted", ""),
    ChromatinParalogPair("SMARCD2", "SMARCD3", "SWI/SNF", "Core", "predicted", ""),
    ChromatinParalogPair("PBRM1", "ARID2", "SWI/SNF (PBAF)", "Scaffold", "published", "29562155"),
    
    # ========== Polycomb Repressive Complex (PRC) ==========
    ChromatinParalogPair("CBX2", "CBX4", "PRC1", "Chromodomain", "predicted", ""),
    ChromatinParalogPair("CBX4", "CBX8", "PRC1", "Chromodomain", "predicted", ""),
    ChromatinParalogPair("CBX7", "CBX8", "PRC1", "Chromodomain", "predicted", ""),
    ChromatinParalogPair("PCGF1", "PCGF2", "PRC1", "RING-finger", "predicted", ""),
    ChromatinParalogPair("PCGF2", "PCGF4", "PRC1 (cPRC1)", "RING-finger", "predicted", ""),
    ChromatinParalogPair("PCGF4", "PCGF5", "PRC1", "RING-finger", "predicted", ""),
    ChromatinParalogPair("RING1", "RNF2", "PRC1", "E3-ligase", "predicted", ""),
    ChromatinParalogPair("RNF2", "RING1", "PRC1", "E3-ligase", "predicted", ""),
    ChromatinParalogPair("EZH1", "EZH2", "PRC2", "Methyltransferase", "published", "31068699"),
    ChromatinParalogPair("EZH2", "EZH1", "PRC2", "Methyltransferase", "published", "31068699"),
    ChromatinParalogPair("SUZ12", "EED", "PRC2", "Core", "predicted", ""),
    
    # ========== NuRD Complex ==========
    ChromatinParalogPair("CHD3", "CHD4", "NuRD", "ATPase", "predicted", ""),
    ChromatinParalogPair("CHD4", "CHD5", "NuRD", "ATPase", "predicted", ""),
    ChromatinParalogPair("MTA1", "MTA2", "NuRD", "Scaffold", "predicted", ""),
    ChromatinParalogPair("MTA2", "MTA3", "NuRD", "Scaffold", "predicted", ""),
    ChromatinParalogPair("HDAC1", "HDAC2", "NuRD/Sin3", "Deacetylase", "published", ""),
    ChromatinParalogPair("HDAC2", "HDAC1", "NuRD/Sin3", "Deacetylase", "published", ""),
    ChromatinParalogPair("GATAD2A", "GATAD2B", "NuRD", "Accessory", "predicted", ""),
    ChromatinParalogPair("MBD2", "MBD3", "NuRD", "MBD", "predicted", ""),
    
    # ========== INO80 Complex ==========
    ChromatinParalogPair("INO80", "SRCAP", "INO80/SRCAP", "ATPase", "predicted", ""),
    ChromatinParalogPair("ACTR5", "ACTR8", "INO80", "Actin-related", "predicted", ""),
    
    # ========== ISWI Complex ==========
    ChromatinParalogPair("SMARCA1", "SMARCA5", "ISWI", "ATPase", "predicted", ""),
    ChromatinParalogPair("SMARCA5", "SMARCA1", "ISWI", "ATPase", "predicted", ""),
    ChromatinParalogPair("BAZ1A", "BAZ1B", "ISWI (ACF/WICH)", "Bromodomain", "predicted", ""),
    ChromatinParalogPair("BAZ2A", "BAZ2B", "ISWI (NoRC)", "Bromodomain", "predicted", ""),
    
    # ========== Histone Acetyltransferases (HAT) ==========
    ChromatinParalogPair("CREBBP", "EP300", "HAT", "Acetyltransferase", "published", "27571770"),
    ChromatinParalogPair("EP300", "CREBBP", "HAT", "Acetyltransferase", "published", "27571770"),
    ChromatinParalogPair("KAT2A", "KAT2B", "SAGA/ATAC", "Acetyltransferase", "predicted", ""),
    ChromatinParalogPair("KAT6A", "KAT6B", "HBO1/MOZ", "Acetyltransferase", "predicted", ""),
    
    # ========== Histone Methyltransferases ==========
    ChromatinParalogPair("KMT2A", "KMT2B", "COMPASS", "H3K4me", "predicted", ""),
    ChromatinParalogPair("KMT2C", "KMT2D", "COMPASS", "H3K4me", "published", ""),
    ChromatinParalogPair("SETD1A", "SETD1B", "COMPASS", "H3K4me", "predicted", ""),
    ChromatinParalogPair("NSD1", "NSD2", "H3K36me", "Methyltransferase", "predicted", ""),
    ChromatinParalogPair("NSD2", "NSD3", "H3K36me", "Methyltransferase", "predicted", ""),
    
    # ========== Histone Demethylases ==========
    ChromatinParalogPair("KDM1A", "KDM1B", "LSD", "H3K4/K9 demethylase", "predicted", ""),
    ChromatinParalogPair("KDM4A", "KDM4B", "JMJD2", "H3K9/K36 demethylase", "predicted", ""),
    ChromatinParalogPair("KDM5A", "KDM5B", "JARID1", "H3K4 demethylase", "predicted", ""),
    ChromatinParalogPair("KDM6A", "KDM6B", "UTX/JMJD3", "H3K27 demethylase", "published", ""),
    
    # ========== Cohesin Complex ==========
    ChromatinParalogPair("STAG1", "STAG2", "Cohesin", "SA subunit", "published", "33007256"),
    ChromatinParalogPair("STAG2", "STAG1", "Cohesin", "SA subunit", "published", "33007256"),
    
    # ========== DNA Methylation ==========
    ChromatinParalogPair("DNMT1", "DNMT3A", "DNA methylation", "Methyltransferase", "predicted", ""),
    ChromatinParalogPair("DNMT3A", "DNMT3B", "DNA methylation", "Methyltransferase", "predicted", ""),
    ChromatinParalogPair("TET1", "TET2", "DNA demethylation", "Dioxygenase", "predicted", ""),
    ChromatinParalogPair("TET2", "TET3", "DNA demethylation", "Dioxygenase", "predicted", ""),
]

print(f"Total paralog pairs in database: {len(CHROMATIN_PARALOG_DATABASE)}")

# 複合体ごとの集計
complex_counts = pd.Series([p.complex_name for p in CHROMATIN_PARALOG_DATABASE]).value_counts()
print("\nPairs by complex:")
print(complex_counts.head(10))

## 4. データローダークラス

In [ ]:
class DepMap25Q3Loader:
    """DepMap Public 25Q3 データローダー"""
    
    FILE_MAPPING = {
        'crispr': 'CRISPRGeneEffect.csv',
        'model': 'Model.csv', 
        'mutations': 'OmicsSomaticMutations.csv',
        'expression': 'OmicsExpressionProteinCodingGenesTPMLogp1.csv',
        'cnv': 'OmicsCNGene.csv',
    }
    
    def __init__(self, data_dir: str = "./depmap_25q3"):
        self.data_dir = Path(data_dir)
        self.crispr: Optional[pd.DataFrame] = None
        self.model: Optional[pd.DataFrame] = None
        self.mutations: Optional[pd.DataFrame] = None
        self.expression: Optional[pd.DataFrame] = None
        self.cnv: Optional[pd.DataFrame] = None
        
    def load_all(self) -> 'DepMap25Q3Loader':
        """全データの一括読み込み"""
        print("Loading DepMap 25Q3 data...")
        self.load_crispr()
        self.load_model()
        self.load_mutations()
        self.load_expression()
        self.load_cnv()
        self._align_samples()
        return self
    
    def load_crispr(self) -> pd.DataFrame:
        """CRISPR依存性スコア"""
        filepath = self.data_dir / self.FILE_MAPPING['crispr']
        print(f"  Loading CRISPR data from {filepath}...")
        self.crispr = pd.read_csv(filepath, index_col=0)
        # カラム名正規化: "TP53 (7157)" → "TP53"
        self.crispr.columns = [c.split(' ')[0] for c in self.crispr.columns]
        print(f"    {self.crispr.shape[0]} cell lines × {self.crispr.shape[1]} genes")
        return self.crispr
    
    def load_model(self) -> pd.DataFrame:
        """細胞株メタデータ"""
        filepath = self.data_dir / self.FILE_MAPPING['model']
        print(f"  Loading Model data from {filepath}...")
        self.model = pd.read_csv(filepath)
        self.model = self.model.set_index('ModelID')
        print(f"    {len(self.model)} cell lines")
        return self.model
    
    def load_mutations(self) -> pd.DataFrame:
        """変異データ"""
        filepath = self.data_dir / self.FILE_MAPPING['mutations']
        print(f"  Loading Mutation data from {filepath}...")
        self.mutations = pd.read_csv(filepath, low_memory=False)
        print(f"    {len(self.mutations)} mutations")
        return self.mutations
    
    def load_expression(self) -> pd.DataFrame:
        """発現データ"""
        filepath = self.data_dir / self.FILE_MAPPING['expression']
        print(f"  Loading Expression data from {filepath}...")
        self.expression = pd.read_csv(filepath, index_col=0)
        self.expression.columns = [c.split(' ')[0] for c in self.expression.columns]
        print(f"    {self.expression.shape[0]} cell lines × {self.expression.shape[1]} genes")
        return self.expression
    
    def load_cnv(self) -> pd.DataFrame:
        """コピー数データ"""
        filepath = self.data_dir / self.FILE_MAPPING['cnv']
        if not filepath.exists():
            print(f"  CNV data not found, skipping...")
            return None
        print(f"  Loading CNV data from {filepath}...")
        self.cnv = pd.read_csv(filepath, index_col=0)
        self.cnv.columns = [c.split(' ')[0] for c in self.cnv.columns]
        print(f"    {self.cnv.shape[0]} cell lines × {self.cnv.shape[1]} genes")
        return self.cnv
    
    def _align_samples(self):
        """サンプルIDの整合性確認"""
        crispr_samples = set(self.crispr.index)
        model_samples = set(self.model.index)
        common = crispr_samples.intersection(model_samples)
        print(f"  Common samples: {len(common)}")

## 5. 交絡補正付き回帰分析クラス

In [ ]:
class ConfounderAdjustedAnalyzer:
    """statsmodels OLS を用いた交絡補正付き解析"""
    
    def __init__(self, loader: DepMap25Q3Loader):
        self.loader = loader
        self.results: List[Dict] = []
        
    def get_lof_status(self, gene: str) -> pd.Series:
        """
        Loss-of-Function 変異ステータスを取得
        Returns: Series with ModelID as index, True/False as values
        """
        if self.loader.mutations is None:
            return pd.Series(dtype=bool)
        
        # Damaging変異のタイプ
        damaging_types = {
            'Nonsense_Mutation', 'Frame_Shift_Del', 'Frame_Shift_Ins',
            'Splice_Site', 'Nonstop_Mutation', 'Start_Codon_Del',
            'De_novo_Start_OutOfFrame', 'Start_Codon_SNP'
        }
        
        gene_muts = self.loader.mutations[
            (self.loader.mutations['HugoSymbol'] == gene) &
            (
                self.loader.mutations['VariantType'].isin(damaging_types) |
                (self.loader.mutations['LikelyLoF'] == True) |
                (self.loader.mutations['isDeleterious'] == True)
            )
        ]
        
        lof_lines = set(gene_muts['ModelID'].unique())
        all_lines = self.loader.crispr.index
        
        return pd.Series(
            [line in lof_lines for line in all_lines],
            index=all_lines,
            name=f'{gene}_LOF'
        )
    
    def get_low_expression_status(self, gene: str, 
                                   percentile: float = 25) -> pd.Series:
        """低発現ステータスを取得"""
        if self.loader.expression is None or gene not in self.loader.expression.columns:
            return pd.Series(dtype=bool)
        
        expr = self.loader.expression[gene]
        threshold = np.nanpercentile(expr, percentile)
        
        return pd.Series(expr <= threshold, name=f'{gene}_low_expr')
    
    def get_cnv_status(self, gene: str, cn_threshold: float = 0.7) -> pd.Series:
        """
        コピー数欠失ステータス
        cn_threshold: log2(CN/2) < -0.7 ≈ CN < 1.2
        """
        if self.loader.cnv is None or gene not in self.loader.cnv.columns:
            return pd.Series(dtype=bool)
        
        cnv = self.loader.cnv[gene]
        return pd.Series(cnv < -cn_threshold, name=f'{gene}_del')
    
    def build_regression_data(self, 
                               driver_gene: str,
                               target_gene: str,
                               stratify_by: str = "mutation") -> pd.DataFrame:
        """
        回帰分析用のデータフレームを構築
        
        Columns:
        - dependency: ターゲット遺伝子のCRISPR依存性スコア
        - driver_status: ドライバー遺伝子の変異/低発現ステータス
        - lineage_*: 組織タイプのOne-hot encoding
        - target_expr: ターゲット遺伝子の発現量
        - target_cnv: ターゲット遺伝子のCNV
        """
        if target_gene not in self.loader.crispr.columns:
            return pd.DataFrame()
        
        # 基本データ
        data = pd.DataFrame({
            'dependency': self.loader.crispr[target_gene],
        })
        
        # ドライバーステータス
        if stratify_by == "mutation":
            data['driver_status'] = self.get_lof_status(driver_gene)
        elif stratify_by == "expression":
            data['driver_status'] = self.get_low_expression_status(driver_gene)
        elif stratify_by == "cnv":
            data['driver_status'] = self.get_cnv_status(driver_gene)
        elif stratify_by == "any":
            lof = self.get_lof_status(driver_gene)
            low_expr = self.get_low_expression_status(driver_gene)
            cnv_del = self.get_cnv_status(driver_gene)
            data['driver_status'] = lof | low_expr | cnv_del
        
        # 組織タイプ（交絡因子）
        if self.loader.model is not None:
            lineage = self.loader.model['OncotreeLineage'].reindex(data.index)
            top_lineages = lineage.value_counts().head(10).index.tolist()
            for lin in top_lineages:
                data[f'lineage_{lin}'] = (lineage == lin).astype(int)
        
        # ターゲット遺伝子の発現量（交絡因子）
        if self.loader.expression is not None and target_gene in self.loader.expression.columns:
            data['target_expr'] = self.loader.expression[target_gene].reindex(data.index)
        
        # ターゲット遺伝子のCNV（交絡因子）
        if self.loader.cnv is not None and target_gene in self.loader.cnv.columns:
            data['target_cnv'] = self.loader.cnv[target_gene].reindex(data.index)
        
        # 欠損値を除去
        data = data.dropna()
        
        return data
    
    def run_ols_analysis(self,
                          driver_gene: str,
                          target_gene: str,
                          stratify_by: str = "mutation",
                          min_affected: int = 5) -> Optional[Dict]:
        """
        OLS回帰による交絡補正付き解析
        Model: dependency ~ driver_status + lineage + target_expr + target_cnv
        """
        data = self.build_regression_data(driver_gene, target_gene, stratify_by)
        
        if len(data) < 20:
            return None
        
        n_affected = data['driver_status'].sum()
        n_unaffected = len(data) - n_affected
        
        if n_affected < min_affected or n_unaffected < min_affected:
            return None
        
        # 回帰式の構築
        covariates = [col for col in data.columns 
                      if col.startswith('lineage_') or col in ['target_expr', 'target_cnv']]
        
        formula = 'dependency ~ driver_status'
        if covariates:
            formula += ' + ' + ' + '.join(covariates)
        
        try:
            # OLS回帰
            model = smf.ols(formula, data=data).fit()
            
            # driver_status の係数を取得
            coef_name = 'driver_status[T.True]' if 'driver_status[T.True]' in model.params else 'driver_status'
            coef = model.params.get(coef_name, np.nan)
            pvalue = model.pvalues.get(coef_name, np.nan)
            se = model.bse.get(coef_name, np.nan)
            
            ci = model.conf_int()
            if coef_name in ci.index:
                ci_lower, ci_upper = ci.loc[coef_name]
            else:
                ci_lower, ci_upper = np.nan, np.nan
            
            # 補正なしの単純比較
            affected_mean = data[data['driver_status'] == True]['dependency'].mean()
            unaffected_mean = data[data['driver_status'] == False]['dependency'].mean()
            raw_delta = affected_mean - unaffected_mean
            
            # 効果量
            std_coef = coef / data['dependency'].std()
            
            return {
                'driver_gene': driver_gene,
                'target_gene': target_gene,
                'stratify_by': stratify_by,
                'n_affected': int(n_affected),
                'n_unaffected': int(n_unaffected),
                'n_total': len(data),
                'affected_mean': affected_mean,
                'unaffected_mean': unaffected_mean,
                'raw_delta': raw_delta,
                'adjusted_coef': coef,
                'adjusted_se': se,
                'adjusted_pvalue': pvalue,
                'ci_lower': ci_lower,
                'ci_upper': ci_upper,
                'standardized_coef': std_coef,
                'r_squared': model.rsquared,
                'n_covariates': len(covariates),
            }
            
        except Exception as e:
            print(f"  Warning: OLS failed for {driver_gene}-{target_gene}: {e}")
            return None

## 6. バッチスキャナークラス

In [ ]:
class ChromatinParalogScanner:
    """クロマチン因子パラログの一括スキャン"""
    
    def __init__(self, loader: DepMap25Q3Loader):
        self.loader = loader
        self.analyzer = ConfounderAdjustedAnalyzer(loader)
        self.results: pd.DataFrame = pd.DataFrame()
        
    def scan_all_pairs(self,
                       paralog_pairs: List[ChromatinParalogPair] = None,
                       stratify_by: str = "any",
                       min_affected: int = 5,
                       verbose: bool = True) -> pd.DataFrame:
        """
        全パラログペアの一括スキャン
        """
        if paralog_pairs is None:
            paralog_pairs = CHROMATIN_PARALOG_DATABASE
        
        print(f"\nScanning {len(paralog_pairs)} chromatin paralog pairs...")
        print(f"Stratification: {stratify_by}")
        print("-" * 60)
        
        results = []
        
        for i, pair in enumerate(paralog_pairs):
            if verbose and (i + 1) % 10 == 0:
                print(f"  Progress: {i+1}/{len(paralog_pairs)}")
            
            result = self.analyzer.run_ols_analysis(
                driver_gene=pair.gene_a,
                target_gene=pair.gene_b,
                stratify_by=stratify_by,
                min_affected=min_affected
            )
            
            if result:
                result['complex'] = pair.complex_name
                result['subunit_type'] = pair.subunit_type
                result['evidence_level'] = pair.evidence_level
                result['pmid'] = pair.pmid
                results.append(result)
        
        df = pd.DataFrame(results)
        
        if len(df) > 0:
            df['fdr'] = false_discovery_control(df['adjusted_pvalue'], method='bh')
            df = df.sort_values('adjusted_pvalue')
            df['significant_fdr05'] = df['fdr'] < 0.05
            df['significant_fdr10'] = df['fdr'] < 0.10
        
        self.results = df
        return df
    
    def scan_by_complex(self, 
                        complex_name: str,
                        stratify_by: str = "any") -> pd.DataFrame:
        """特定の複合体のみスキャン"""
        pairs = [p for p in CHROMATIN_PARALOG_DATABASE 
                 if complex_name.lower() in p.complex_name.lower()]
        return self.scan_all_pairs(pairs, stratify_by=stratify_by)
    
    def get_summary_by_complex(self) -> pd.DataFrame:
        """複合体ごとの結果サマリー"""
        if len(self.results) == 0:
            return pd.DataFrame()
        
        summary = self.results.groupby('complex').agg({
            'driver_gene': 'count',
            'significant_fdr05': 'sum',
            'adjusted_coef': 'mean',
            'adjusted_pvalue': 'min',
        }).rename(columns={
            'driver_gene': 'n_pairs_tested',
            'significant_fdr05': 'n_significant',
            'adjusted_coef': 'mean_effect',
            'adjusted_pvalue': 'best_pvalue',
        })
        
        return summary.sort_values('n_significant', ascending=False)

## 7. 可視化関数

In [ ]:
def plot_volcano(results: pd.DataFrame, 
                 title: str = "Chromatin Paralog Synthetic Lethality Scan",
                 figsize: tuple = (12, 8)):
    """Volcano plot"""
    fig, ax = plt.subplots(figsize=figsize)
    
    df = results.copy()
    df['neg_log_p'] = -np.log10(df['adjusted_pvalue'])
    
    colors = []
    for _, row in df.iterrows():
        if row['fdr'] < 0.05 and row['adjusted_coef'] < -0.1:
            colors.append('red')
        elif row['fdr'] < 0.1 and row['adjusted_coef'] < -0.1:
            colors.append('orange')
        else:
            colors.append('lightgray')
    
    ax.scatter(df['adjusted_coef'], df['neg_log_p'],
               c=colors, alpha=0.6, s=60, edgecolors='black', linewidths=0.5)
    
    sig_df = df[df['fdr'] < 0.1].nsmallest(15, 'adjusted_pvalue')
    for _, row in sig_df.iterrows():
        ax.annotate(
            f"{row['driver_gene']}→{row['target_gene']}",
            (row['adjusted_coef'], row['neg_log_p']),
            fontsize=8, fontweight='bold' if row['fdr'] < 0.05 else 'normal'
        )
    
    ax.axhline(-np.log10(0.05), color='blue', linestyle='--', alpha=0.5, label='p=0.05')
    ax.axvline(-0.1, color='green', linestyle='--', alpha=0.5, label='coef=-0.1')
    
    ax.set_xlabel('Adjusted Coefficient (driver LOF → target dependency)', fontsize=12)
    ax.set_ylabel('-log10(adjusted p-value)', fontsize=12)
    ax.set_title(title, fontsize=14)
    ax.legend()
    
    plt.tight_layout()
    plt.show()


def plot_forest(results: pd.DataFrame, n_top: int = 20, figsize: tuple = (10, 8)):
    """Top hitsのフォレストプロット"""
    df = results.nsmallest(n_top, 'adjusted_pvalue').copy()
    df = df.sort_values('adjusted_coef')
    
    fig, ax = plt.subplots(figsize=figsize)
    
    y_pos = np.arange(len(df))
    
    xerr = np.array([
        df['adjusted_coef'] - df['ci_lower'],
        df['ci_upper'] - df['adjusted_coef']
    ])
    
    colors = ['red' if fdr < 0.05 else 'orange' if fdr < 0.1 else 'gray' 
              for fdr in df['fdr']]
    
    ax.errorbar(df['adjusted_coef'], y_pos, xerr=xerr,
                fmt='o', color='black', ecolor='gray', capsize=3, markersize=8)
    ax.scatter(df['adjusted_coef'], y_pos, c=colors, s=100, zorder=5)
    
    ax.axvline(0, color='black', linestyle='-', alpha=0.3)
    
    labels = [f"{row['driver_gene']}→{row['target_gene']} ({row['complex']})" 
              for _, row in df.iterrows()]
    ax.set_yticks(y_pos)
    ax.set_yticklabels(labels)
    
    ax.set_xlabel('Adjusted Coefficient (95% CI)', fontsize=12)
    ax.set_title(f'Top {n_top} Chromatin Paralog Synthetic Lethality Hits', fontsize=14)
    
    plt.tight_layout()
    plt.show()


def plot_complex_heatmap(results: pd.DataFrame, figsize: tuple = (14, 10)):
    """複合体ごとのヒートマップ"""
    pivot = results.pivot_table(
        values='adjusted_coef',
        index='driver_gene',
        columns='target_gene',
        aggfunc='mean'
    )
    
    fig, ax = plt.subplots(figsize=figsize)
    
    sns.heatmap(pivot, cmap='RdBu_r', center=0, annot=True, fmt='.2f', ax=ax,
                cbar_kws={'label': 'Adjusted Coefficient'})
    
    ax.set_title('Chromatin Factor Paralog Dependencies\n(Adjusted for Lineage, Expression, CNV)', fontsize=14)
    ax.set_xlabel('Target Gene (dependency)', fontsize=12)
    ax.set_ylabel('Driver Gene (LOF)', fontsize=12)
    
    plt.tight_layout()
    plt.show()


def plot_single_pair(results: pd.DataFrame, driver: str, target: str, loader: DepMap25Q3Loader):
    """個別ペアの詳細プロット"""
    analyzer = ConfounderAdjustedAnalyzer(loader)
    data = analyzer.build_regression_data(driver, target, stratify_by="any")
    
    if len(data) == 0:
        print(f"No data for {driver} - {target}")
        return
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    # Violin plot
    ax1 = axes[0]
    plot_data = pd.DataFrame({
        'Dependency': data['dependency'],
        'Group': data['driver_status'].map({True: f'{driver} LOF', False: f'{driver} WT'})
    })
    sns.violinplot(data=plot_data, x='Group', y='Dependency', ax=ax1)
    sns.stripplot(data=plot_data, x='Group', y='Dependency', color='black', alpha=0.3, size=3, ax=ax1)
    ax1.axhline(-1, color='red', linestyle='--', alpha=0.5)
    ax1.set_title(f'{target} Dependency')
    ax1.set_ylabel('CRISPR Dependency Score')
    
    # Box plot
    ax2 = axes[1]
    sns.boxplot(data=plot_data, x='Group', y='Dependency', ax=ax2)
    ax2.axhline(-1, color='red', linestyle='--', alpha=0.5, label='Strong dependency')
    ax2.set_title(f'{target} Dependency Distribution')
    ax2.legend()
    
    plt.suptitle(f'{driver} → {target} Synthetic Lethality', fontsize=14)
    plt.tight_layout()
    plt.show()
    
    # 統計情報
    affected = data[data['driver_status'] == True]['dependency']
    unaffected = data[data['driver_status'] == False]['dependency']
    t_stat, p_val = stats.ttest_ind(affected, unaffected)
    
    print(f"\n{driver} LOF: n={len(affected)}, mean={affected.mean():.3f}")
    print(f"{driver} WT:  n={len(unaffected)}, mean={unaffected.mean():.3f}")
    print(f"Delta: {affected.mean() - unaffected.mean():.3f}")
    print(f"t-test p-value: {p_val:.2e}")

## 8. デモデータ生成（実データがない場合）

In [ ]:
def generate_demo_data() -> DepMap25Q3Loader:
    """DepMap 25Q3形式のデモデータ生成"""
    np.random.seed(42)
    
    n_lines = 1000
    cell_lines = [f"ACH-{str(i).zfill(6)}" for i in range(n_lines)]
    
    # 遺伝子リスト
    chromatin_genes = list(set(
        [p.gene_a for p in CHROMATIN_PARALOG_DATABASE] +
        [p.gene_b for p in CHROMATIN_PARALOG_DATABASE]
    ))
    other_genes = [f"GENE{i}" for i in range(300)]
    genes = chromatin_genes + other_genes
    
    # 組織タイプ
    lineages = np.random.choice(
        ['Lung', 'Breast', 'Colon', 'Skin', 'Brain', 'Kidney', 'Ovary', 'Pancreas', 'Liver', 'Stomach'],
        n_lines, p=[0.15, 0.12, 0.12, 0.08, 0.08, 0.1, 0.08, 0.1, 0.08, 0.09]
    )
    
    # CRISPRデータ
    crispr = pd.DataFrame(
        np.random.normal(-0.05, 0.25, (n_lines, len(genes))),
        index=cell_lines, columns=genes
    )
    
    # モデル情報
    model = pd.DataFrame({
        'OncotreeLineage': lineages,
        'OncotreePrimaryDisease': [f'{lin} Cancer' for lin in lineages],
    }, index=cell_lines)
    
    # 発現データ
    expression = pd.DataFrame(
        np.random.normal(5, 2, (n_lines, len(genes))),
        index=cell_lines, columns=genes
    )
    
    # CNVデータ
    cnv = pd.DataFrame(
        np.random.normal(0, 0.3, (n_lines, len(genes))),
        index=cell_lines, columns=genes
    )
    
    # 変異データ（合成致死をシミュレート）
    mutations = []
    
    # SMARCA4 LOF → SMARCA2依存性
    smarca4_lof = cell_lines[:80]
    for line in smarca4_lof:
        mutations.append({'ModelID': line, 'HugoSymbol': 'SMARCA4',
                         'VariantType': np.random.choice(['Nonsense_Mutation', 'Frame_Shift_Del']),
                         'LikelyLoF': True, 'isDeleterious': True})
    crispr.loc[smarca4_lof, 'SMARCA2'] = np.random.normal(-1.0, 0.3, len(smarca4_lof))
    expression.loc[smarca4_lof, 'SMARCA4'] = np.random.normal(1, 0.5, len(smarca4_lof))
    
    # ARID1A LOF → ARID1B依存性
    arid1a_lof = cell_lines[100:160]
    for line in arid1a_lof:
        mutations.append({'ModelID': line, 'HugoSymbol': 'ARID1A',
                         'VariantType': np.random.choice(['Nonsense_Mutation', 'Frame_Shift_Del']),
                         'LikelyLoF': True, 'isDeleterious': True})
    crispr.loc[arid1a_lof, 'ARID1B'] = np.random.normal(-0.8, 0.3, len(arid1a_lof))
    expression.loc[arid1a_lof, 'ARID1A'] = np.random.normal(1.5, 0.5, len(arid1a_lof))
    
    # STAG2 LOF → STAG1依存性
    stag2_lof = cell_lines[200:250]
    for line in stag2_lof:
        mutations.append({'ModelID': line, 'HugoSymbol': 'STAG2',
                         'VariantType': np.random.choice(['Nonsense_Mutation', 'Frame_Shift_Del']),
                         'LikelyLoF': True, 'isDeleterious': True})
    crispr.loc[stag2_lof, 'STAG1'] = np.random.normal(-0.7, 0.35, len(stag2_lof))
    
    # CREBBP LOF → EP300依存性
    crebbp_lof = cell_lines[400:440]
    for line in crebbp_lof:
        mutations.append({'ModelID': line, 'HugoSymbol': 'CREBBP',
                         'VariantType': np.random.choice(['Nonsense_Mutation', 'Frame_Shift_Del']),
                         'LikelyLoF': True, 'isDeleterious': True})
    crispr.loc[crebbp_lof, 'EP300'] = np.random.normal(-0.9, 0.25, len(crebbp_lof))
    
    # EZH2 LOF → EZH1依存性
    ezh2_lof = cell_lines[300:340]
    for line in ezh2_lof:
        mutations.append({'ModelID': line, 'HugoSymbol': 'EZH2',
                         'VariantType': 'Nonsense_Mutation',
                         'LikelyLoF': True, 'isDeleterious': True})
    crispr.loc[ezh2_lof, 'EZH1'] = np.random.normal(-0.6, 0.3, len(ezh2_lof))
    
    # ランダム変異
    for _ in range(1000):
        mutations.append({
            'ModelID': np.random.choice(cell_lines),
            'HugoSymbol': np.random.choice(chromatin_genes),
            'VariantType': np.random.choice(['Missense_Mutation', 'Silent']),
            'LikelyLoF': False,
            'isDeleterious': np.random.choice([True, False], p=[0.2, 0.8]),
        })
    
    mutations = pd.DataFrame(mutations)
    
    # ローダー作成
    loader = DepMap25Q3Loader()
    loader.crispr = crispr
    loader.model = model
    loader.expression = expression
    loader.cnv = cnv
    loader.mutations = mutations
    
    return loader

print("Demo data generator ready!")

---
## 9. 解析実行

### 9.1 データの読み込み

In [ ]:
# 実データを使用する場合は USE_REAL_DATA = True に変更
USE_REAL_DATA = False

if USE_REAL_DATA:
    # 実データの読み込み
    loader = DepMap25Q3Loader(data_dir=DATA_DIR)
    loader.load_all()
else:
    # デモデータの生成
    print("[Demo Mode] Generating simulated data...")
    loader = generate_demo_data()

print(f"\n✅ Data loaded:")
print(f"   Cell lines: {len(loader.crispr)}")
print(f"   Genes: {loader.crispr.shape[1]}")
print(f"   Mutations: {len(loader.mutations)}")

### 9.2 一括スキャンの実行

In [ ]:
# スキャナーの初期化
scanner = ChromatinParalogScanner(loader)

# 一括スキャン実行
# stratify_by: "mutation", "expression", "cnv", "any"（OR条件）
results = scanner.scan_all_pairs(
    stratify_by="any",  # 変異 OR 低発現 OR CNV欠失
    min_affected=5,
    verbose=True
)

print(f"\n✅ Scan complete!")
print(f"   Total pairs tested: {len(results)}")
print(f"   Significant (FDR < 0.05): {results['significant_fdr05'].sum()}")
print(f"   Significant (FDR < 0.10): {results['significant_fdr10'].sum()}")

### 9.3 結果の確認

In [ ]:
# Top 15 hits
print("\n" + "="*70)
print("TOP 15 HITS (by adjusted p-value)")
print("="*70)

display_cols = ['driver_gene', 'target_gene', 'complex', 'n_affected', 
                'raw_delta', 'adjusted_coef', 'adjusted_pvalue', 'fdr']

results.head(15)[display_cols]

In [ ]:
# 複合体別サマリー
print("\n" + "="*70)
print("SUMMARY BY COMPLEX")
print("="*70)

complex_summary = scanner.get_summary_by_complex()
complex_summary

### 9.4 可視化

In [ ]:
# Volcano plot
plot_volcano(results, title="Chromatin Paralog Synthetic Lethality Scan\n(OLS Confounder Adjusted)")

In [ ]:
# Forest plot
plot_forest(results, n_top=15)

In [ ]:
# SWI/SNF複合体のヒートマップ
swisnf_results = results[results['complex'].str.contains('SWI/SNF', na=False)]
if len(swisnf_results) > 3:
    plot_complex_heatmap(swisnf_results)

In [ ]:
# 個別ペアの詳細プロット（例: SMARCA4 → SMARCA2）
plot_single_pair(results, 'SMARCA4', 'SMARCA2', loader)

### 9.5 結果の保存

In [ ]:
# CSVとして保存
results.to_csv('chromatin_paralog_scan_results.csv', index=False)
complex_summary.to_csv('chromatin_paralog_complex_summary.csv')

print("Results saved to:")
print("  - chromatin_paralog_scan_results.csv")
print("  - chromatin_paralog_complex_summary.csv")

# Google Driveに保存する場合
if USE_GOOGLE_DRIVE:
    results.to_csv('/content/drive/MyDrive/chromatin_paralog_scan_results.csv', index=False)
    print("  - Also saved to Google Drive")

---
## 10. カスタム解析

### 10.1 特定の複合体のみスキャン

In [ ]:
# SWI/SNFのみ
swisnf_results = scanner.scan_by_complex("SWI/SNF", stratify_by="mutation")
swisnf_results[['driver_gene', 'target_gene', 'adjusted_coef', 'adjusted_pvalue', 'fdr']]

### 10.2 カスタムパラログペアの追加

In [ ]:
# カスタムペアを定義
custom_pairs = [
    ChromatinParalogPair("YOUR_GENE_A", "YOUR_GENE_B", "Custom", "Custom", "user_defined"),
    # 追加のペア...
]

# スキャン実行
# custom_results = scanner.scan_all_pairs(custom_pairs, stratify_by="any")

### 10.3 単一遺伝子ペアの詳細解析

In [ ]:
# 特定のペアを詳細解析
analyzer = ConfounderAdjustedAnalyzer(loader)

# 例: ARID1A → ARID1B
result = analyzer.run_ols_analysis(
    driver_gene="ARID1A",
    target_gene="ARID1B",
    stratify_by="any"
)

if result:
    print(f"\nARID1A → ARID1B Analysis:")
    print(f"  N (affected): {result['n_affected']}")
    print(f"  N (unaffected): {result['n_unaffected']}")
    print(f"  Raw delta: {result['raw_delta']:.4f}")
    print(f"  Adjusted coef: {result['adjusted_coef']:.4f}")
    print(f"  95% CI: [{result['ci_lower']:.4f}, {result['ci_upper']:.4f}]")
    print(f"  Adjusted p-value: {result['adjusted_pvalue']:.2e}")
    print(f"  R-squared: {result['r_squared']:.4f}")

---
## 11. まとめ

### 主な発見
- **SMARCA4 → SMARCA2**: SWI/SNF ATPaseパラログ（最も強いシグナル）
- **ARID1A → ARID1B**: SWI/SNF DNA結合サブユニット
- **CREBBP → EP300**: ヒストンアセチルトランスフェラーゼ
- **STAG2 → STAG1**: コヒーシン複合体

### 次のステップ
1. 実データ（DepMap 25Q3）での検証
2. 組織特異的な解析
3. 実験的バリデーション（CRISPRスクリーン）
4. ドラッグターゲットとしての評価